In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'    # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

In [ ]:
import ast

df_raw = pd.read_csv('../../../data/raw/steam_indie_list.csv')

df_raw['total_reviews'] = df_raw['positive'] + df_raw['negative']
df_raw['release_date']  = pd.to_datetime(df_raw['release_date'], errors='coerce')
df_raw['positive_rate'] = df_raw['positive'] / df_raw['total_reviews'] * 100

def parse_owners_lower(s):
    try:
        return int(s.split('..')[0].strip().replace(',', ''))
    except Exception:
        return 0

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df_raw['owners_lower'] = df_raw['owners'].apply(parse_owners_lower)
df_raw['genres'] = df_raw['genres'].apply(parse_genres)

print(f'모집단: {len(df_raw):,}개')
print(f'출시연도 분포:')
print(df_raw['release_date'].dt.year.value_counts().sort_index())



In [ ]:
# 전체 장르 별 게임 수 현황
genre_counts = {}
for genres in df_raw['genres']:
    for genre in genres:
        genre_counts[genre] = genre_counts.get(genre, 0) + 1

# 장르별 분포 DataFrame 생성
genre_df = pd.DataFrame(list(genre_counts.items()), columns=['Genre', 'Count']).sort_values('Count', ascending=False)

display(genre_df)

In [ ]:
# 통계 확인
genre_df.describe()

In [ ]:
# 3. 게임 수의 중앙값 및 평균 확인
median_games = genre_df['Count'].median()
mean_games = genre_df['Count'].mean()

print(f"--- 장르별 게임 수 통계 ---")
print(f"장르당 게임 수 중앙값: {median_games:.1f}개")
print(f"장르당 게임 수 평균값: {mean_games:.1f}개")
print(f"최소 게임 수: {genre_df['Count'].min()}개")
print(f"최대 게임 수: {genre_df['Count'].max()}개")

# 4. 중앙값 미만인 장르 목록 확인 (필터링 검토용)
minor_genres = genre_df[genre_df['Count'] < median_games].sort_values('Count')
print(f"\n--- 중앙값({median_games}) 미만 장르 (총 {len(minor_genres)}개) ---")
print(minor_genres['Genre'].tolist())


In [ ]:
# 장르 개수 중앙값 계산
median_count = genre_df['Count'].median()

print(f'장르별 게임 수 중앙값: {median_count:.0f}')
print(f'평균: {genre_df["Count"].mean():.1f}')
print(f'표준편차: {genre_df["Count"].std():.1f}')
print(f'최소값: {genre_df["Count"].min()}')
print(f'최대값: {genre_df["Count"].max()}')

# 중앙값보다 많은 장르와 적은 장르 분류
above_median = genre_df[genre_df['Count'] >= median_count]
below_median = genre_df[genre_df['Count'] < median_count]

print(f'\n중앙값 이상의 장르: {len(above_median)}개')
print(f'중앙값 미만의 장르: {len(below_median)}개')

print(f'\n=== 중앙값 이상 장르 ({len(above_median)}개) ===')
print(above_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))

# 중앙값 미만 장르 출력
print(f'\n=== 중앙값 미만 장르 ({len(below_median)}개) ===')
print(below_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))

In [ ]:
import pandas as pd

# 리스트 내에서 'Indie'가 있으면 제외하고 새로운 리스트를 생성.
df_raw['genres'] = df_raw['genres'].apply(lambda x: [genre for genre in x if genre != 'Indie'])

# 장르 중 early_access 여부 컬럼 추가
df_raw['is_early_access'] = df_raw['genres'].apply(
    lambda x: any('early access' in g.lower() for g in x)
)

# 장르 중 free to play 여부 컬럼 추가
df_raw['is_f2p'] = df_raw['genres'].apply(
    lambda x: any('free to play' in g.lower() for g in x)
)

# 장르 중 massively multiplayer 여부 컬럼 추가
df_raw['is_massively_multiplayer'] = df_raw['genres'].apply(
    lambda x: any('massively multiplayer' in g.lower() for g in x)
)

# 결과 확인 (Early Access 게임 수)
print(f"Early Access 게임 수: {df_raw['is_early_access'].sum()}개")
print(f"f2p 게임 수: {df_raw['is_f2p'].sum()}개")
print(f"massively multiplayer 게임 수: {df_raw['is_massively_multiplayer'].sum()}개")

# 1. 기본 필터링 (2023~2025년, EA 제외, F2P 제외, massively multiplayer 장르 제외)
df_filtered = df_raw[
    (df_raw['release_date'].dt.year >= 2023) &
    (df_raw['release_date'].dt.year <= 2025) &
    (~df_raw['is_early_access']) &
    (~df_raw['is_f2p']) &
    (~df_raw['is_massively_multiplayer'])
].copy()

# 3. 장르별 게임 수 집계 (Indie 제외 후)
genre_counts = df_filtered['genres'].explode().value_counts()

# 4. 게임 수가 80개를 초과하는 '유효 장르' 리스트 추출
# (중앙값 기준 혹은 요청하신 80개 기준 적용)
valid_genres = genre_counts[genre_counts > 80].index.tolist()

# 5. 각 게임의 genres '유효 장르'만 남기도록 업데이트
df_filtered['genres'] = df_filtered['genres'].apply(
    lambda x: [genre for genre in x if genre in valid_genres]
)

# 6. 유효 장르가 하나도 없는 게임은 분석 대상에서 최종 제외
df_final = df_filtered[df_filtered['genres'].map(len) > 0].copy()

# --- 결과 확인 ---
print(f"필터링 전 게임 수: {len(df_raw):,}개")
print(f"최종 분석 대상 게임 수: {len(df_final):,}개")
print(f"\n선정된 유효 장르 ({len(valid_genres)}개):")
print(valid_genres)

# 최종 데이터 확인
df_final[['genres']]


In [ ]:
# 최종본 데이터 확인
df_filtered.head()